# Regenerate s1K reasoning and answers with DeepSeek Pro

Run on **Colab CPU**. Add `HF_TOKEN` (read access) and `DEEPSEEK_API_KEY` to Colab Secrets
and enable notebook access. This notebook contains its helper code: no Git push or GPU
is required. Use a fresh runtime when opening a new version of this notebook.

Source: `Seungjun/dp_removed_s1K-1.1`, pinned to the same **996 rows** used by the project
before the SFT length filter. Every original column and row is preserved in the full export.

DeepSeek Pro solves the original **question only**; it is not given the old reasoning,
answer, reference solution, or grade. Thinking is enabled. Its native `reasoning_content`
becomes `deepseek-v4-pro_reasoning`; its final `content` becomes `deepseek-v4-pro_answer`.
The prompt encourages Kimi k1.5's **planning, evaluation, reflection, and exploration**
where useful, with **no short-reasoning instruction or target length**. These are reasoning
practices, not mandatory headings. There is **no answer-correctness filter**.

1. Run setup, settings, and source-loading cells.
2. Run **Smoke test** for 20 reproducibly random examples and download the five-column CSV.
3. Use the separate **Full run** cell to generate the entire dataset, reusing completed rows.

A default **Run all** runs the paid smoke test only; the full-run checkbox starts off.
Both modes save each completed response immediately to Google Drive. Rerun a cell after
interruption to resume; unsuccessful rows are attempted again, completed rows are reused.
Use only **one notebook/runtime at a time** for the same output folder.

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/LG-AIME-S1-DeepSeek"

In [ ]:
# @title Install CPU dependencies, mount Drive, and read secrets
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets==5.0.1", "huggingface-hub==0.36.2", "httpx==0.28.1",
    "PyYAML==6.0.3", "tqdm==4.67.1"])
from google.colab import drive, userdata, files
drive.mount("/content/drive")
HF_TOKEN = userdata.get("HF_TOKEN")
DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
if not HF_TOKEN or not DEEPSEEK_API_KEY:
    raise ValueError("Enable notebook access to HF_TOKEN and DEEPSEEK_API_KEY in Secrets")

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-s1-regeneration")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = 'UEsDBBQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAY29tbW9uL19faW5pdF9fLnB5AwBQSwMEFAAAAAgAAAAhXHQtxyXcAgAAhggAAAwAAABjb21tb24vaW8ucHnVVUtv2zAMvudXEDrJa2psBbZDgOyybodh6AbscemKQI7pRY0jGZLcJCj630dKTuLEwbDDephOJmk+vk8kJYS4bp0qaoSPXz/fQKVr9KBMCapp0JSX1tRbuLetM6r2sNZhAQ7n9gGTkzYBnWubgCUEpWufCyFGI71qrAuwUH5R62In3ntrdt/WjypnV9CowL9Ap/5C4mg0KrGCUv9CH+SDqlvM4PIt+OAmI6DjMFA9u+i5X6ir129kNPHhNHnZrhqfnMfgKfRsiVs//eZYRuNbhzPl51pPPxAwzHI0c1uizGKYLF/gpisg6+pZOx1wxsElFz2GVFkqiTUwjeVHa7bX5o1yaEK+WpbaySTs69hoH2Z2GcXkEpCJUG5L0aI7Uz7zbVXpTYycp2+4AJGHVSOSW7yYvW9u6e6kWIsMlIdqMqRmx0x1jowxXWtJZU6vsr1nlUf8Uvw0oq+t6tYv5EFjfV75rZlLMlEvGSuzE2C5w6ZWc+x4OiW37th1du3/L3Ir67hqIi8WfzD0+es1J/10thU5+/OwnGb6lOZ/zXKkK/olptSQqecnIwF2qI7hviDE2CjtZrytuhx/wq8rMDYkOBGrl9kBSLeJbu/SXqJbpwidtKJ1QpJwF4XgML28gJSWLIU4Rxg7njK2XhAyYI6P28oHRVtzSqwErOseKXxqbTDamAYWTuwdNrYcR+VTkNPySBvcdvgbN/w07ZXaqtJLjnacBjdzbAKk2+ZX5hp51b53zroxfDeapZ4uG2Y5oY/fJ648IZOvsr2Gs9MqLz1zKovYOsNwfIgx15q5ojaMJGZn/xqyEEErTff3g1dorFhW4p2Nz2B8RT+BClBsA8JjDP00gUe+3icahPjo3ViDR0G5c/I0nTwJg2s6B/7voFa5R1zKl2O4GiLczWFxGLGuo7mgboZqoohGvNjOdJkG4+gVJoiOa74VuhR32SR2RH8Vno5gf/7i0nga/QZQSwMEFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAABwaXBlbGluZS9fX2luaXRfXy5weQMAUEsDBBQAAAAIAAAAIVxaN0gI8hAAAKU4AAAZAAAAcGlwZWxpbmUvcmVnZW5lcmF0ZV9zMS5web0baW/cNva7fwXLfqgmlRWnF5pJtUDapNsg7TZIsrtYDAaCrOGMWWtEVaR81PB/3/ceSYk6xuN2gfUHe4bH4+O7D5pz/oOqirZpRGXYKyHqD0JcskbkWlWy2rGdqESTG6mqF6xSLK/0tWhOCwUbClMJrdlWlkY0sDbhnJ+cyH2tGsNyfVsVUvmvhapvu8/6yn9s8mqj9v6bkXtxsm3Unm1yI/AbczP+e0xr/lCVWyf2uSyT1shS+6V13miB6zOjMr/Prq5zc1HKc7/yHXzt8L0wpr45sesKtd+ryi+DSwSjiVR+Iq9rUW2y34BSZcw2cie0iZF03dh1IwEP/BJ+Lk9OTjZiS1fRJt/X0WJ5wuCnEaZtqu6ySaWuI39fuGSxSKRWW9XscxMtHJCrvJR0W63aphBRo651DLhWW7lzYHUtCpa6sRW3C/ma5uSWlaKiXQv2SUprV1zcwB8jNhmO8zVTjbveeCHANCA4mZ3la3sg3SWXWrB/5WUrXjeNaiL+gc4FQNutaEBskKbmQrBaVpXYsI1AYPleVnCbDdPP3iIhci0MXxBUuDiD45ms8I/uj8KJKkdhqbpLFqps9xXgnlwhCtqT2P/AvSsFvNWyAhZUlnDJTpgIIS1ipk2zwHvjKpha4fA6gVFZj2HNXnfLf5FaowZZejMjbsyS3SGce+65VyoQFsc5izpIuLoUlTvCKwOSoZNw2uQGT45y2C2E6XBfZPnXiFrxdUzkSzuWAhAc1HUpjRulzzA4uXhIA3EF9FRV6mHbrwiK7pTam1lRBw4iShJkyqFkJx6U51BLcMJREUaa2yzfghnKNIrRRkfEdkdFx2waCiTUwjlLzmgIQEzm9vlNBPMx2wLtjINpsRA3hahNwPF+8wDSHLRZE+WgJ4FVYKdkI2gocucGZ0f94TH7eFv7j79eiQYQvqavi1lU8NInJ0WZg/3+Echfvnz3xsrt+7bC48K9YNZftqCpMFGQJ4jZuSxLlG1QEFkRywDy76Bnhgncqdm+hc/aqBpuC7PXqrm0/sGd+l70ngXOtCchM7NMVtJkWaRFuY1ZyP+YqdbUrQGzpExwMVyZOIlCR5NswI/hB2uuhussqMnKUMLm5LA7Iw7B9OvzWgLMYGrFYcjpoBNCGFjxvdqIEmwqmZ+K3XFEQYPTPb366rRuFI9ZP7QFWl3w+xEPJ6b1n5Vua7QNYDg7H24PWkwQ6Hx7JrZgOk2IC0gNnn8hdxf4FyT2EYfD95z1CJgLWV2icDjwAwxGRpcQKnwAUtyirZCVIcs7nWPfsWfHsAk2WBk8R+xqpaWRV+giDAhecxwpuHqGdkUKPUFqMAdInR1DKtgQIFUpcHy7/GG8nrHv0v5UsqB0aMq+fP7lF8++8T7qLHn+dbcU1C6r7aqjBHvj9LeTGzqDFUJ6DbfQ5gSJND5DewGK6U2vPfesp5asMr8Ur9nAeeHa4+R7MzIxcCCgNqXVWUcA6xHQkMjhYYN5pGg393gkYOsUhUKB4RxbAL0HWvI1BlRg0EsUlwfkjkD0otZx34KGD12k9lhc6Xym6Wym5R+Cj4xhbwtswMTSoZH6nPGsW8OHe20q8MBGu4AP5Ka6jeYPtlEd3nsGuJscRYBHyfB362HAJrlwkOUlhue34EEh8HgBvNy2FKMZxRR4TYrRfcC2laIEwRhRjEgKkrWRhUB/YzOY5D39iea5D6Z8w9eLxDIiGmFd7UTUcXYRW3b3p37Kfq3KW+YibQjVBCAKsVyRl5TQgEM6F6WqdhrpggF1I3S7z8+B5XKDTtvcJj0PEIM9jJLjBfxdXL/CZCjKMkjmRJYtEspiMGSNACOakyqZnV4HauiO68He+XByGSoGeJYhHjA9HLgfyyk4fIBKiIQxAHvKeNOCRYQPA/l72iGzWj77Zj2FluwvN7KJIBSDZTr92LSQXJJYZOqSvvYo7PNKbtHwpOxuwDzuD8ELuI9wub9252FszQeC5mENBkcbKnGd+bRnyVazahbPaNcoqOdBbp/Z3B7A/ZiXWowWAsEzFO2s8/3Thffdpz7/jXqWPsUYwxI3wSkIOvz3kQT8BkpZgcinbLA7N0bsa6Npd8nH1g3usiEt7ZLyKISG2TqEp01mclmOmE7rSrAelu/9RcgKEWRUuPCgoT0KAKzsghUHzokbvl4TSjg0PC4DWaaqS2rpeNJFxbotCiC12LiwmAAtJglLcCYls7QsBuwX9JWDpzEt2DSWpsholD8jeHBMfiV84E34BUeE5Y4JFWntLPUTuy+aXXOUPLScakk2ObjOITfQpTKEQYgdrbmW5sKTEoN3MJOXQ67kCMFXpxJdQtQTdYmZ3VmBXfMRi0/B9grcMWhSAXnYYsrm4Z50sgf84TQ1WE2S6SOx0oQabqnjWFFKYQOIoWwcTU3O1eZ2atusKV0O7CpYMR/ew8wdN5B1wgcuKvQ3G34/MhGTXGN5KAdByBRnLgch7AheEAMvp1ExwNAGQO8PGCxIrHW+I2s6pf0db1RJl9G3GowKt4Ycne7Qkq84ZGlgc/j6floO6aG0GgznFAZ6+RVxaIb/nTxNylgrTpymcsr43PWcuUUr5YwjxUsUZsxkL5+zZ4s5DbFi3avbSOhNTrneVNJH1QYykyBaTrWXzFkknzyA95iVswN1Jm4hwq4cKRrUSuLOF4DiZBAcEUhHALjk/QgxjP/hqBwF35eAOtphRaSzweHMpLRDBOtNj7crLiWKHs6TZuqIFjldqwpi2dTxwip2AkksmPCnxUVunjrbDfKgQcjQJqeoxYsDAK15xRp35lyAtbH2oMSOQTywEQ9ilFCMnYFsOTDR/IFY0Qvhk9efLrV4JW2NhZbowZMzuUkRqnVjcsMXcT9HQhNMu8rHvBBZ7YbIBhSiqRuwssHO6SQe1KLVCFbR9zn40ysWFwpCNQyIYTfoM31FfT5bT9Y684SlKVq28gaLT9d2BjR2LRnY5VZbHMPIzxoguMhgRTc+AQ7Xl/qiw8Mut4MuG5zZ5KMJ2DbPSg81RSutag6IbyjJHeENKVlezhyAP7glrNn3VKB6PU6fK1X2E75i/yhwlpJjWHb0MKC/KNcOwXTCydSjMaB4ar/Ny7QjWjpPyQN6QAqc9lEg5ucdEwVYPshYqm72McJOVpUSz9Rzm6qKQOB2u5UFWrHMqRgor+1SoOE+t9nD0Ea7Kjc15pKfPn5894EwptweDC3OT+2nvRWcD7PJo+xbiLTbjUh/dfZtzL46ew6/vnh+T3UJO/m3lH19djbVGeczqF/kQU5WfcpeKVpSql1XzrqA5EQ0WOVuBCWR4HZiKp8159I0eXML3rjBrJ9q6xiuge9Ojoid5y9mabQPKE1/0y1HarI7u+J+RtWG7nGuvTIgr7uAFztYfUqr+Yy2gJB5HqVI2nkX+NhofsD6uUAcQ/vpRDwXoOPSh9pbIVXiB8uMo2v7do2V5I8QimlMl33jxkYFtuFyUK5DKcU89S8zHyN25N4iyTLs+mXZPLJhb+mtuHWf3mD0Nu04vTRgHM9bI/5ft+C/5CU2wsXmvRPCJQcWHrucD4SAhbXGwkVXGAYpVy1kqRO5OPWxbsy+nEn8KGEep7b4A3JOJuFQq3jYddvyrvxuddNh+lkQsn22vn8BHlkUl0NDwR8UWv/Dz/MSnRz5Na8fWhgDngeAYHsPhpu2StgPzuZvbJcWr7hJ+OR6PTMRpA+103SmR3KICGGzMepJYFBF0FmcWoPnWyfi5iJvMWd40Tkqh+OjaMDzRrjbgCBjk9qwYf3IEQeogBVhqqOKmZsPbPyslLdNNRg+z4tLtd2iQZJVdKgF8bBVYU/YF+zJE4/yEKtP2U8gtG3D3pP5fYnml4krYOo1chaSCFRssdEMF4kb0BkrQKceNdfwGbqVQ9UStykObSJYvVF9wnV4RV+zorIl3jK7FGBEn8QMcukdENpXYI23juk/AMMgQeKcvyd+2FKY1kK/cL2YbS5LLwiqYjnDqhI1ppXtVyfse1Ru24kOGoTUkA74OqjBzTUXhvL68UJqhrIi+8ZCXmA7b9qaclc+1rF49fr1uw+vX7/NwDJkb1//B0GjskpQ+BBo136gBxTu6+BQ7Cho0c/Rox0c9AMUYFS3kcQOHDXyMbSx21ydYkFVBInxkN/12M4Tvhhq6xIfDHQNFV/LHtUAg6LnwCeYXF/iFVdBoSrHPAclo181zsjtu6LfN/skb033VgsHToaifawqFiDogw/MyJ0u/Axfx0WRuQBkXFvAWqi9rAQhBeqPSewlxrqXruwrR0GFJQVeKzIKXEka8hZ1jaxHChoTTSBN2XrwXc9G6CLtLXP/ZgNccQtnpL61yfEFnEaDmCL2Xq2HSAdhnQ2GXuLAD1TemMZe5znk9m1TpmQV/Tc0ky7gTO/oYYpq5B++pbLl3wsw8w27cxp3P65JktDYmCt9uDwTWqPu0xRYKffS6NRe6Gf6gjYS8+4KXAyWaNKZdw6P8toI5xKIDzp1JY5AHDGRojBbOTpUskIbjQ9zQA4PlKJIUDDmQ2Fxons4Ag8rh740PSxKH9wK4uyjvmcPrwKjlmElbCtvIh8F/M9iPvmB5BnE9yKlEuDiZCpB3jh5ewAhGT3ogvHI09RikPX119nju4hg9A4mVOjFYloCmq1F4s/Qae9yiO2a6AlhPFvrycvyACREH/dRZxn3H+Y9TicFRpjlgaLgA2jFLmbKbPZBAj7sxs2iCZYSzSD4SDQ56Bam+KHIFKXSYtZYH+y3YYTUmHHggoXFYUzy+sY+BS5L1lW03Zs1MnNVCzO2Y62JnBistCBaFGpq0OyGPKRNibG9rsdRCR7av9Cyjwog6t8C5Me8DLus1HXl7kOw/lwcQadTzY4OtiUzt+2TdKYrfQyjD/QmxeODj6FaWIHvFkCjkXj0ksI9WuHzuNDdx6jQFfrnFH0UszgatfyIXApRqoFDWHLBEBrLLzaAcW9P3DfsQ4QPtiBN3InuCZBrUO/DkpNd8WdeHli5EKMwKG93+GqA+i/B8MBYz3PD92MO94kH64Oa7ti6Ts15h5Xv+N5NlPHJk3EPbGqRZ98tLLvUvX+NtJ4WLNEEHIA4ePLQg3NvlB4Fa0Qcz5vuuu7ddTZudbmOiz900ImHaedWD9Vpu59RBX4IblSdPwbKVlOGIOzY0a1BR2YEIJx5PBjf/puF5Do5AeUb/JcPZ7PR68+GylPP32tUu99jRRV7kp3FhuOH4XNfFR/OsNMegdkb8m6aruSXskA23Hue2Uc8HsrhNzh9jRxuTf+kQco5Y7xb10xaNYOuFaZmd/f2Fcv4AQuVdbpnGG6WDlmPCWj/dwALY679DlcryhYlOegujx8VYFFuSXxru3+2iNkZIXVGOLV4KqE61Lbwnzx8B943/nt+QeQQDFJi1L0O6MDNvkly1vspcMpe0L9Hcl8XM5vKcBfps3uGFHfG4SFfuhzZ2lKPE9Ku/T8hInkE3BI+DIiZHYGEaIfxUvB0czrnLd8BRX3887FhegthHealHV2QMXkjAUxS6KthnQ7rWKqx2ogbEwyZMuoV3UQc1ydmX4/qbxRWdTsTBeYz4vhqHeJmheKX8tZsT7891XLH8bnkdSkrkXJOCdF2GiESPzGZxvNeycL8mwaibWyfgBK5U/qNfhoyQZ1TEpZyuatUI2YaKBZmQn9ssjoTFIeL0CVGnQddzJMJ+FGXeSEiJFbw7yBgFpYPciHi7n9sEuAF1gWcnNoyrjeotufnV9oVI9oHoo9nxGwGZfcU7Y7jCjBbdqFXK770GtV5xswtPKBM9yf/BVBLAwQUAAAACAAAACFcMTBLrvoDAACOBwAAIwAAAGNvbmZpZ3MvcmVnZW5lcmF0ZV9zMV9kZWVwc2Vlay55YW1sdVVNbyM3DL3nVxDosYkz/o6npwW67aFFd9FtbwUMWeJ4lGikqaRxYrT73/uo8Vda9GKMJPKRfI+kUxii5vqOKHIfavrCg98/D/7R9NvIXTiw2abpTw/TybTYHGyywde0MtP1etesKjXdLI2e69luMTe8Uk/cNNPZolqqqd4sZnDSwTd2X5PhRg0u4yb1zuaaclTW48hvPeuMQDG8ppo2m9XoldnnrbF7TjBerZeLWbPkZj5rNrxu5utlpea7DULxclGt1mbNel1Vi+lqpvjJzJWeVbPNajPbqGY1X97p4IbOJyn1zwGQpYzzFy5DtHvrlUPZKgVvfUmZ+8T8ss2t9S+42iLnZyQb4vHWRfn0yvHGXuXMXZ/vVG8l4E4l3g7R1dTm3Kf68REPk7P1RIcORl0w7K4YD4fFQx9DYf2U0BbchggyWrtv8ZBDv+1rmk4qHL6hTz1HJdUoR5qtgwfB/upP39KY6T398uk3UngpBLChrOKeMzn2+9xOCtyH0+nB2c6KTeTUB5+YbKLGqf2ezT15PnCElKzERCWgopzecWYIq+RL4Dr1ts3hhSEATefTan3qDD3EyF4fcSuqd9ZvT0ltLRogHsBuYhgaOFaTWaFjfM+24zDk6/P0qapOoSLnaBl38+KQ4xFwNttbtNnlSTwu16vqLnXItC6dWQoQY0GGLKYmNDV0gbo1/f2A2y/BHZhyy5SGHp0NGvC+c9yR9fTR751N7T10PbALvchwQ9FVm17lFvKAQ2qkqU5SCXm/p4Kf3plHpbPVnOi15Vjej9Sy6yld8jmlIZU80Gen/NjV1mCwbHMsNpEdH5TP1AAv3YsmqQymHMZ0enjSjtFKIinrIUsCNk8K7kdINKhxnHQEx1o5dyTdsn4h2/VoWIEvanZsLPoEXJhBi0sqIeChB1cw0gj6KzeO9QgaRZmEpGMxjgx6GJ+kehSodCsMeCG1hMSOYuWkDzubsnphjCkOhpUh9uaU9FvvQjxnfYF3yNHjFvydwUeC/Q2/GIPC8OA1ozYUdpQQnYSCzx7vpYoPaIbjSberWoAfojD03QipQKoPGe7eKNkrCKIMCE5j3qUodFDGtMuOxpwJ+I/2pHGHtmH8CO30PGCbNfiUwjCabGS0Q2lM6EBHbPsydW6QNT6h70MJbv0BLXEmLJXV9n9sdOg2qSuA0m5sFiiqLiVO6Gcskpv2g51IL6LhFtiqCwOihebazhP6BDbHIcKeLuljy2DM1U7GR1aLNIqQIVsH0AdIVhz+PS2fh/yfe1QoEtMfu/DG5q+vF00vSbZldRmbdJS5fAdKPxQpYBqakRv8ZWEoRprP/yHvs7qu1uImTlfiZTFcUzzv1cndP1BLAQIUAxQAAAAIAAAAIVwAAAAAAgAAAAAAAAASAAAAAAAAAAAAAACAAQAAAABjb21tb24vX19pbml0X18ucHlQSwECFAMUAAAACAAAACFcdC3HJdwCAACGCAAADAAAAAAAAAAAAAAAgAEyAAAAY29tbW9uL2lvLnB5UEsBAhQDFAAAAAgAAAAhXAAAAAACAAAAAAAAABQAAAAAAAAAAAAAAIABOAMAAHBpcGVsaW5lL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAAAhXFo3SAjyEAAApTgAABkAAAAAAAAAAAAAAIABbAMAAHBpcGVsaW5lL3JlZ2VuZXJhdGVfczEucHlQSwECFAMUAAAACAAAACFcMTBLrvoDAACOBwAAIwAAAAAAAAAAAAAAgAGVFAAAY29uZmlncy9yZWdlbmVyYXRlX3MxX2RlZXBzZWVrLnlhbWxQSwUGAAAAAAUABQBUAQAA0BgAAAAA'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)

## Settings and generation prompt

Defaults come from the bundled `configs/regenerate_s1_deepseek.yaml`, shown below.
Edit `CONFIG` in this cell before loading the source if needed. The default is
**`deepseek-v4-pro`, high thinking effort, 16 concurrent requests** with request pacing
and bounded retries for rate limits and transient API failures. Temperature is omitted
because DeepSeek ignores it in thinking mode. The sampling seed fixes the 20 selected
examples; hosted generation itself is not guaranteed deterministic.

`max_tokens=131072` is an operational ceiling for **reasoning + answer**, not a requested
length or an SFT limit. It is not included in the prompt. A response that hits this ceiling
is flagged as incomplete; its raw text is saved in the journal, but is not placed in the
completed-output columns. No source rows are removed by the old 20,480-token SFT limit.

Changing the config or bundled generation code creates a separate run folder. Reuse the
same notebook/settings to resume. Hosted model aliases may change server-side: returned
model IDs, timestamps, fingerprints, usage, and response IDs are recorded per request.

In [ ]:
import yaml
CONFIG = yaml.safe_load((CODE_ROOT / "configs/regenerate_s1_deepseek.yaml").read_text())
# Optional edits go here, before creating the run in the next cell.
# CONFIG["api"]["concurrency"] = ...
print(yaml.safe_dump(CONFIG, sort_keys=False, allow_unicode=True))

In [ ]:
# @title Load the pinned Hugging Face source and prepare a resumable run
import importlib.metadata
from pipeline.regenerate_s1 import RegenerationRun, load_source, timestamp
from common.io import append_jsonl, read_jsonl
source_rows = load_source(CONFIG, HF_TOKEN)
run = RegenerationRun(source_rows, CONFIG, PROJECT_ROOT)
runtime_path = run.root / "runtime_sessions.jsonl"
read_jsonl(runtime_path, repair_tail=True)
append_jsonl(runtime_path, {"started_at": timestamp(), "python": sys.version,
    "packages": {name: importlib.metadata.version(name) for name in
                 ["datasets", "huggingface-hub", "httpx", "PyYAML", "tqdm"]}})
print("Source rows:", len(source_rows))
print("New columns:", run.reasoning_column, run.answer_column)
print("Smoke sample indices (seeded):", run.smoke_indices)
print("Drive output folder:", run.root)

## Smoke test — 20 random examples and a comparison CSV

This cell makes paid API requests for the 20 sampled rows that are not already complete.
The CSV has exactly these five columns, in this order:

`question`, `deepseek_thinking_trajectory`, `deepseek_attempt`,
`deepseek-v4-pro_reasoning`, `deepseek-v4-pro_answer`.

UTF-8 CSV preserves multiline text, quotation marks, and Unicode. Long reasoning can exceed
spreadsheet apps' cell-display limits; the CSV itself retains the full text. Blank generated
cells mean generation is incomplete/failed/pending; inspect `smoke/status.jsonl` for details.
The old `deepseek_grade` is not a grade for the new answer, and no grading is run here.

In [ ]:
# @title Smoke test: generate the fixed random sample and save/download the CSV
try:
    await run.generate(run.smoke_indices, DEEPSEEK_API_KEY)
finally:
    smoke_result = run.export(run.smoke_indices, "smoke")
    print(smoke_result["summary"])
    print("CSV saved:", smoke_result["path"])
    print("Row status:", smoke_result["status_path"])
files.download(str(smoke_result["path"]))

## Full run — all 996 rows, preserving original columns

Enable the checkbox below to run. There is no baseline-evaluation or smoke-completion gate.
Matching completed smoke results are reused automatically. After a runtime restart, run
setup/settings/source-loading, then this cell directly to resume.

The export is a Hugging Face-compatible **JSONL dataset** in original source order, with
every original field plus the two generated fields. Existing grades and responses retain
their original meaning; they are never overwritten. The source HF repository is unchanged.

`full/dataset.jsonl` is written when every row is complete. Otherwise the export is called
`full/dataset.partial.jsonl`, still containing every row, with null generated fields for
unsuccessful rows. `full/status.jsonl` lists failures by source index; `attempts.jsonl`
retains raw returned text, including incomplete responses and retry usage. Rerun the cell
to retry remaining rows. Authentication/balance/invalid-request errors stop the run and
save partial results. Failed responses are not silently promoted into the training data.

This notebook only regenerates data. It does not run SFT, evaluation, or HF publication.

In [ ]:
# @title Full run: generate all rows (completed smoke rows are reused)
RUN_FULL_GENERATION = False # @param {type:"boolean"}
if RUN_FULL_GENERATION:
    full_indices = list(range(len(source_rows)))
    try:
        await run.generate(full_indices, DEEPSEEK_API_KEY)
    finally:
        full_result = run.export(full_indices, "full")
        print(full_result["summary"])
        print("Dataset saved:", full_result["path"])
        print("Row status:", full_result["status_path"])
    if full_result["summary"]["remaining"]:
        print("Some rows remain incomplete. Inspect status.jsonl and rerun this cell.")
    else:
        print("All source rows now have generated reasoning and answers.")
else:
    print("Full run is off. Enable RUN_FULL_GENERATION to process all source rows.")

Load the finished export later with
`load_dataset("json", data_files=".../full/dataset.jsonl", split="train")`.
Drive stores the dataset and journals; save a notebook copy in Drive to retain cell output
displays too. The API journal records returned token usage, including retries. A request
whose response was lost or cancelled can still be billed; consult the provider dashboard.

References: [Kimi k1.5, §2.2](https://arxiv.org/html/2501.12599v1#S2.S2),
[DeepSeek thinking fields](https://api-docs.deepseek.com/guides/thinking_mode/),
[DeepSeek API](https://api-docs.deepseek.com/api/create-chat-completion/).